# Milestone 2 (Week 4) — Transit Reliability KPIs
**Urban Pulse: Smart City Mobility Intelligence Platform**

Upload these 5 files first:
- `stop_times_clean.csv`, `trips_clean.csv`, `routes_clean.csv`, `route_zone_mapping.csv` (from the transit cleaning step)
- `fact_traffic.csv` (from the merge step)

**Honest scope of this notebook:**
- Average Travel Time and Average Speed are calculated from REAL schedule data
- "Estimated Demand Index" is a clearly-labeled proxy for Passenger Count, NOT a real ridership count
- On-Time Performance is marked as unavailable, since no real-time tracking data exists in any collected file

In [9]:
# MILESTONE 2: Transit Reliability KPIs
# Urban Pulse - Smart City Mobility Intelligence Platform

In [10]:
import pandas as pd
import numpy as np
import os

os.chdir(r"C:\infosys internship task\given data")

stop_times = pd.read_csv("stop_times_clean.csv")
trips = pd.read_csv("trips_clean.csv")
routes = pd.read_csv("routes_clean.csv")
route_zone = pd.read_csv("route_zone_mapping.csv")
fact_traffic = pd.read_csv("fact_traffic.csv")

In [11]:
# KPI 1 & 2: Average Travel Time and Average Speed (REAL data)

In [12]:
def time_to_minutes(t):
    """Convert HH:MM:SS to minutes, handling GTFS times that can exceed 24:00:00"""
    try:
        h, m, s = map(int, t.split(":"))
        return h * 60 + m + s / 60
    except Exception:
        return np.nan

stop_times["arrival_min"] = stop_times["arrival_time"].apply(time_to_minutes)

# Per trip: travel time = last stop time - first stop time
trip_times = stop_times.groupby("trip_id")["arrival_min"].agg(["min", "max"])
trip_times["travel_time_min"] = trip_times["max"] - trip_times["min"]

# Per trip: total distance = sum of est_distance across its stops
trip_distance = stop_times.groupby("trip_id")["est_distance"].sum().rename("distance_km")

trip_perf = trip_times.join(trip_distance)
trip_perf = trip_perf[trip_perf["travel_time_min"] > 0]  # drop bad/zero-duration trips
trip_perf["avg_speed_kmph"] = trip_perf["distance_km"] / (trip_perf["travel_time_min"] / 60)

# Attach route_id to each trip
trip_perf = trip_perf.join(trips.set_index("trip_id")[["route_id"]], how="left")

# Aggregate to route level
route_perf = trip_perf.groupby("route_id").agg(
    avg_travel_time_min=("travel_time_min", "mean"),
    avg_speed_kmph=("avg_speed_kmph", "mean"),
    trip_count=("travel_time_min", "count"),
).reset_index()

print(f"Routes with real travel time / speed calculated: {len(route_perf)}")
print(route_perf.describe())

Routes with real travel time / speed calculated: 70
       avg_travel_time_min  avg_speed_kmph  trip_count
count            70.000000       70.000000   70.000000
mean            100.525476        7.818572    1.071429
std              50.327428       10.651212    0.353919
min              24.000000        0.000000    1.000000
25%              56.937500        0.000000    1.000000
50%              87.183333        0.000000    1.000000
75%             138.604167       11.162654    1.000000
max             246.333333       44.869177    3.000000


In [13]:
# KPI 3: Passenger Count — ESTIMATE, clearly labeled (no real data exists)

In [14]:
# We do NOT have ticketing/ridership data anywhere. As an honest proxy,
# we use "Public Transport Usage" (%) from the traffic dataset, averaged
# by Zone, as a stand-in "Estimated Demand Index" — NOT a real passenger count.

demand_by_zone = fact_traffic.groupby("Zone")["Public Transport Usage"].mean().rename(
    "estimated_demand_index"
).reset_index()

route_perf = route_perf.merge(route_zone, on="route_id", how="left")
route_perf = route_perf.merge(demand_by_zone, on="Zone", how="left")

print(f"\nRoutes with a Zone (needed for the demand-index estimate): "
      f"{route_perf['Zone'].notna().sum()} / {len(route_perf)}")


Routes with a Zone (needed for the demand-index estimate): 70 / 70


In [15]:
# KPI 4: On-Time Performance — NOT CALCULATED (no real data exists)

In [16]:
# GTFS static files only contain the SCHEDULE, never actual arrival times.
# Calculating real on-time performance requires live GPS/AVL tracking data,
# which does not exist in any file collected for this project.
# This is intentionally left out rather than faked.
route_perf["on_time_performance"] = "Data not available (requires real-time tracking data)"

In [17]:
# Save output

In [18]:
route_perf.to_csv("fact_transit_performance.csv", index=False)
print("\nSaved: fact_transit_performance.csv")
print(route_perf.head(10))


Saved: fact_transit_performance.csv
  route_id  avg_travel_time_min  avg_speed_kmph  trip_count             Zone  \
0        1           162.850000        7.537708           2  Bangalore South   
1      100            34.638889        8.543930           3   Bangalore East   
2    BIAS1           197.400000       18.058814           1  Bangalore South   
3   BIAS10           109.383333       26.328943           1   Bangalore West   
4   BIAS11           130.583333       26.274569           1  Byatarayanapura   
5   BIAS12           164.850000       35.908394           1  Byatarayanapura   
6    BIAS4           163.994444       17.967499           3  Byatarayanapura   
7    BIAS5           153.283333       18.822177           1  Byatarayanapura   
8    BIAS6           246.333333       14.745039           1  Byatarayanapura   
9    BIAS7           108.833333       44.869177           1  Byatarayanapura   

   estimated_demand_index                                on_time_performance  
0  

In [19]:
import os
print([f for f in os.listdir() if f.startswith("fact_") or f.startswith("dim_")])

['dim_date.csv', 'dim_zone.csv', 'fact_traffic.csv', 'fact_transit_performance.csv', 'fact_transit_routes.csv', 'fact_transit_stops.csv']
